# Task 4.2 — EEGNet regression decoder (braindecode)

EEGNet on 1 kHz LFP predicting Index fingertip velocity per 50 ms window,
trained on stimulation-OFF trials.

- **4.2.1** Adapt braindecode's EEGNet to regression: one velocity pair per 50 ms bin.
- **4.2.2** Split the data into training days, validation days and test days.
- **4.2.3** Early stopping on the validation days.
- **4.2.4** Sweep the kernel length, dropout and depth multiplier.
- **4.2.5** Train 5 seeds with the best setting and compare held-out OFF R² with ridge (task 3.1).
- **4.2.6** Retrain with synthetic stimulation tones in the training data.
- **4.2.7** Average the 5 seeds into an ensemble.
- **4.2.8** Compare every method per day, because sessions on one day share a recording.
- **4.2.9** Separate the clean-input result from the stimulated-input result.

## Setup

In [ ]:
import os
import shutil
import sys
import warnings

import pandas as pd
import torch
from torch import nn
from braindecode.models import EEGNet

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
if TASK4_DIR not in sys.path:
    sys.path.insert(0, TASK4_DIR)

from utils import (
    TASK4_RESULTS, RESULTS_DIR, TRAINING_LOG_DIR, ENSEMBLE_SEED,
    EEGNET_METHOD, EEGNET_TONE_METHOD, EEGNET_ENSEMBLE_METHOD,
    add_stimulation_tones, best_eegnet_settings, bootstrap_confidence_interval,
    day_level_summary, eegnet_sweep_settings, off_training_datasets,
    off_validation_datasets, save_checkpoint, save_training_log, score_ensemble,
    score_trial_model, scores_per_day, stimulated_test_datasets,
    sweep_eegnet_settings, train_with_early_stopping, write_baseline_parquet,
)

warnings.simplefilter('ignore', DeprecationWarning)

SEEDS = (0, 1, 2, 3, 4)
FIGURE_TRAINING_LOGS = os.path.join(RESULTS_DIR, 'training_logs')

### 4.2.1 Adapting braindecode EEGNet to regression

braindecode's `EEGNet` is built for classification: it pads symmetrically and pools the time
axis away, which is useless for regression that needs one prediction per moment. Two changes
turn it into a per-bin regressor:

- `final_conv_length=1` with `n_outputs=2` keeps one (vx, vy) pair per time step instead of
  flattening the axis away; pooling 2 then 25 maps 2000 samples onto 40 bins of 50 ms.
- Every symmetric padding becomes an explicit left-only `ConstantPad2d` before an unpadded
  convolution, so a bin only ever reads samples at or before its own time — the decoder
  stays causal.

In [ ]:
class CausalEEGNet(nn.Module):
    """braindecode EEGNet with left-only padding and one (vx, vy) per 50 ms bin."""

    def __init__(self, channel_count=35, samples=2000, outputs=2, temporal_filters=8,
                 depth_multiplier=2, kernel_samples=500, separable_samples=62,
                 pool_first=2, pool_second=25, dropout=0.5):
        super().__init__()
        self.bin_count = samples // (pool_first * pool_second)

        backbone = EEGNet(n_chans=channel_count, n_outputs=outputs, n_times=samples,
                          final_conv_length=1, F1=temporal_filters, D=depth_multiplier,
                          kernel_length=kernel_samples,
                          depthwise_kernel_length=separable_samples,
                          pool1_kernel_size=pool_first, pool2_kernel_size=pool_second,
                          drop_prob=dropout)

        layers = []
        for _, module in backbone.named_children():
            if isinstance(module, nn.Conv2d) and module.padding[1] > 0:
                # EEGNet pads both sides; a causal decoder may only look backwards
                layers.append(nn.ConstantPad2d((module.kernel_size[1] - 1, 0, 0, 0), 0.0))
                module.padding = (module.padding[0], 0)
            layers.append(module)
        self.layers = nn.Sequential(*layers)

    def forward(self, lfp):
        return self.layers(lfp).transpose(1, 2)


### 4.2.2 Training days, validation days and test days

The split is by *day*, never by trial, so nothing leaks across recordings. `TRAIN` is every
stimulation-off training session; `VALIDATION` the three stimulation-off validation days
from `splits.json`, which early stopping and the sweep both watch; `TEST` pairs each
stimulated test session with the same day's stimulation-off recording — that pairing is
what produces the clean-input score beside the stimulated-input score later. Five test days
carry the fourteen stimulated sessions, so a day counts once when methods are compared.

In [ ]:
TRAIN = off_training_datasets()
VALIDATION = off_validation_datasets()
TEST = stimulated_test_datasets()

print('training trials  ', TRAIN['features'].shape)
print('validation trials', VALIDATION['features'].shape)
print('test sessions    ', len(TEST), '| days', len({session.split('-')[1] for session in TEST}))


### 4.2.3 Early stopping on the validation days

`train_with_early_stopping` keeps the weights from the epoch with the best validation R²
and stops once validation stops improving, so each run spends as many epochs as it needs.
The loss is NaN-masked: bins without kinematics simply do not contribute.

In [ ]:
def train_seeds(train, setting, method, seeds=SEEDS):
    models, summaries, rows = [], [], []
    for seed in seeds:
        torch.manual_seed(seed)  # train_with_early_stopping seeds after construction
        model, history = train_with_early_stopping(CausalEEGNet(**setting), train, VALIDATION, seed)
        models.append(model)
        summaries.append(dict(seed=seed, best_val_r2=history['best_val_r2'],
                              best_epoch=history['best_epoch'],
                              stopped_epoch=history['stopped_epoch']))
        rows += [score_trial_model(model, session, method, 'raw', seed, query, off)
                 for session, (query, off) in TEST.items()]
        save_checkpoint(model, f'task4_2_{method}_seed{seed}')
        save_training_log(history, method, seed, TRAINING_LOG_DIR)
        print(method, 'seed', seed, 'validation R2', round(history['best_val_r2'], 4))
    return models, pd.DataFrame(rows), pd.DataFrame(summaries)

### 4.2.4 Sweeping the kernel length, dropout and depth multiplier

One knob at a time around the default, plus two combinations, one seed each; the best mean
validation R² wins. Since the sweep and the early stopping both look at the validation days,
the selected setting is optimistic *there* — but the test days stay untouched, so the
comparison in 4.2.5 is clean.

In [ ]:
SWEEP = sweep_eegnet_settings(eegnet_sweep_settings(), TRAIN, VALIDATION, CausalEEGNet)
SWEEP.to_csv(os.path.join(TASK4_RESULTS, 'eegnet_sweep.csv'), index=False)

BEST_SETTINGS = best_eegnet_settings(SWEEP)
print('selected settings:', BEST_SETTINGS)
SWEEP.round(4)

### 4.2.5 Five seeds with the best setting

Each seed is early-stopped on the validation days and scored on every stimulated test
session. Five seeds because the seed-to-seed spread is ±0.03 — one seed would not say
anything.

In [ ]:
models, BASELINE_SCORES, SEED_SUMMARY = train_seeds(TRAIN, BEST_SETTINGS, EEGNET_METHOD)
SEED_SUMMARY.to_csv(os.path.join(TASK4_RESULTS, 'task4_2_seeds.csv'), index=False)
BASELINE_SCORES.groupby('session')[['r2_off_ave', 'r2_ave']].mean().round(3)

### 4.2.6 Retraining with synthetic stimulation tones

The baseline never saw stimulation, so stimulated input is outside what it was fitted on.
This arm adds 3 Hz and 20 Hz tones to the *training* trials — same architecture, same test
days — to ask whether stimulation-tolerant input helps. Its row sits beside the baseline
and never replaces it.

In [ ]:
TONED_TRAIN = add_stimulation_tones(TRAIN, seed=SEEDS[0])
_, TONE_SCORES, _ = train_seeds(TONED_TRAIN, BEST_SETTINGS, EEGNET_TONE_METHOD)
TONE_SCORES.groupby('session')[['r2_off_ave', 'r2_ave']].mean().round(3)


### 4.2.7 Seed ensemble

Averaging the five seeds' predictions before scoring takes the seed-to-seed spread out of
the comparison. The averaged rows carry `seed = ENSEMBLE_SEED` (-1) so a groupby on `seed`
cannot mistake them for a sixth run of seed 0.

In [ ]:
ENSEMBLE_SCORES = pd.DataFrame([
    score_ensemble(models, session, EEGNET_ENSEMBLE_METHOD, 'raw', query, off,
                   seed=ENSEMBLE_SEED)
    for session, (query, off) in TEST.items()])

ENSEMBLE_SCORES[['session', 'seed', 'r2_off_ave', 'r2_ave']].round(3)


### 4.2.8 Comparing methods per day

Sessions from one day share a single stimulation-off recording, so they score identically
against it. Averaging per day counts each recording once and leaves five independent
days — the honest unit of comparison.

In [ ]:
SCORES = pd.concat([BASELINE_SCORES, TONE_SCORES, ENSEMBLE_SCORES], ignore_index=True)
SCORES.to_csv(os.path.join(TASK4_RESULTS, 'task4_2_eegnet_off.csv'), index=False)
write_baseline_parquet(SCORES, os.path.join(RESULTS_DIR, 'results_eegnet.parquet'))

os.makedirs(FIGURE_TRAINING_LOGS, exist_ok=True)
for method in (EEGNET_METHOD, EEGNET_TONE_METHOD):
    for seed in SEEDS:
        shutil.copy(os.path.join(TRAINING_LOG_DIR, f'{method}_seed{seed}.json'),
                    FIGURE_TRAINING_LOGS)

print('wrote', len(SCORES), 'rows for', SCORES.method.nunique(), 'methods')


In [ ]:
RIDGE = pd.read_csv(os.path.join(RESULTS_DIR, 'task3', 'task3_1_test_results_lmp_bands_lags9.csv'))
RIDGE = (RIDGE.assign(day=RIDGE.session.str.split('-').str[1])
         .groupby('day', as_index=False).r2_off.mean().rename(columns={'r2_off': 'ridge'}))

COMPARISON = (scores_per_day(SCORES, 'r2_off_ave')
              .query('method == @EEGNET_METHOD')
              .rename(columns={'r2_off_ave': 'eegnet'})
              .merge(RIDGE, on='day'))
COMPARISON['eegnet_minus_ridge'] = COMPARISON.eegnet - COMPARISON.ridge
COMPARISON.to_csv(os.path.join(TASK4_RESULTS, 'task4_2_vs_ridge.csv'), index=False)

low, high = bootstrap_confidence_interval(COMPARISON.eegnet_minus_ridge)
print(f'days {len(COMPARISON)} | eegnet {COMPARISON.eegnet.mean():+.3f} | '
      f'ridge {COMPARISON.ridge.mean():+.3f} | difference '
      f'{COMPARISON.eegnet_minus_ridge.mean():+.3f} (95% CI {low:+.3f} to {high:+.3f})')
COMPARISON.round(3)


In [ ]:
BASELINES = {'knn': ('results_ridge_knn.parquet', 'knn_off'),
             'mlp': ('results_mlp.parquet', 'mlp'),
             'gru': ('results_gru.parquet', 'gru')}


def baseline_by_day(method):
    filename, label = BASELINES[method]
    frame = pd.read_parquet(os.path.join(RESULTS_DIR, filename))
    frame = frame[frame.method == label].assign(day=lambda f: f.session.str.split('-').str[1])
    return (frame.groupby('day', as_index=False).r2_off_ave.mean()
            .rename(columns={'r2_off_ave': method}).set_index('day'))


TABLE = COMPARISON[['day', 'ridge']].set_index('day')
TABLE = TABLE.join(scores_per_day(SCORES, 'r2_off_ave')
                   .pivot(index='day', columns='method', values='r2_off_ave'))
for method in BASELINES:
    TABLE = TABLE.join(baseline_by_day(method))
TABLE.reset_index().to_csv(os.path.join(TASK4_RESULTS, 'task4_2_baseline_off.csv'), index=False)
TABLE.round(3)


### 4.2.9 Clean input versus stimulated input

The same checkpoints score well on each test day's stimulation-off recording and poorly on
the stimulated session itself. That gap *is* the domain shift this project is about, so the
two numbers are reported apart rather than averaged into one misleading score.

In [ ]:
CLEAN = day_level_summary(SCORES, 'r2_off_ave').assign(input='clean')
STIMULATED = day_level_summary(SCORES, 'r2_ave').assign(input='stimulated')

pd.concat([CLEAN, STIMULATED], ignore_index=True).round(4)
